In [4]:
!pip install nina_funcs

In [1]:
import os
import glob
import numpy as np
from scipy.io import savemat
import pywt
import pandas as pd
from sklearn.model_selection import KFold

# Import functions from your ninapro preprocessing library.
from nina_funcs import (
    get_data,
    normalise,
    filter_data,
    notch_filter,
    windowing,
    get_categorical,
    rectify
)

In [2]:
def process_subject_file_kfold(file_path, gestures, win_len, win_stride, n_splits=5):
    """
    Process a single subject file using K-Fold cross validation.

    Parameters:
      file_path (str): Full path to the subject's .mat file.
      gestures (list): List of gesture labels to classify.
      win_len (int): Window length (adjusted for sampling rate).
      win_stride (int): Window stride (adjusted for sampling rate).
      n_splits (int): Number of folds for cross validation.
    
    Returns:
      folds: A list of tuples (X_train, y_train_cat, X_test, y_test_cat) for each fold.
    """
    data_path = os.path.dirname(file_path)
    file_name = os.path.basename(file_path)
    
    # Load raw data.
    data = get_data(data_path, file_name)

    # Use all available repetitions for normalization and windowing.
    # Assuming the subject has repetitions 1 through 6.
    all_reps = [1, 2, 3, 4, 5, 6]
    data = normalise(data, all_reps)
    
    # Create windows from all available data.
    X, y, _ = windowing(data, all_reps, gestures, win_len, win_stride)
    
    # Apply K-Fold splitting.
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = []
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        y_train_cat = get_categorical(y_train)
        y_test_cat = get_categorical(y_test)
        folds.append((X_train, y_train_cat, X_test, y_test_cat))
    
    return folds

In [3]:
def main():
    # Define your input and output directories.
    input_folder = r'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset'
    output_folder = r'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2'
    os.makedirs(output_folder, exist_ok=True)
    
    # Find subject files matching pattern (e.g. S1_E1_A1.mat, S2_E1_A1.mat, etc.).
    subject_files = glob.glob(os.path.join(input_folder, 'S*_E1_A1.mat'))
    if not subject_files:
        print("No subject files found in the specified directory.")
        return
    
    # Define processing parameters.
    gestures = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17]
    # For a 300ms window at 2000Hz: win_len = 300ms * 2 = 600; for a 10ms stride: win_stride = 10ms * 2 = 20.
    # (Adjust these as necessary; here, we'll keep your original values.)
    win_len = 300
    win_stride = 200
    
    # Set the number of folds for cross-validation.
    n_splits = 5
    
    # Process each subject file using k-fold cross validation.
    for file_path in subject_files:
        print(f"Processing file: {file_path}")
        folds = process_subject_file_kfold(file_path, gestures, win_len, win_stride, n_splits)
        
        # Build output filename for each fold.
        base_name = os.path.splitext(os.path.basename(file_path))[0]  # e.g., "S1_E1_A1"
        subject_id = base_name.split('_')[0]  # "S1"
        
        for fold_idx, (X_train, y_train_cat, X_test, y_test_cat) in enumerate(folds, start=1):
            output_filename = f"{subject_id}_E1_A1_fold{fold_idx}_300_200_N.mat"
            output_filepath = os.path.join(output_folder, output_filename)
            
            # Create dictionary with keys expected by your model code.
            out_data = {
                'train_data': X_train,
                'train_labels': y_train_cat,
                'test_data': X_test,
                'test_labels': y_test_cat
            }
            
            savemat(output_filepath, out_data)
            print(f"Saved fold {fold_idx} preprocessed data to: {output_filepath}")

if __name__ == "__main__":
    main()

Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset\S10_E1_A1.mat
Saved fold 1 preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S10_E1_A1_fold1_300_200_N.mat
Saved fold 2 preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S10_E1_A1_fold2_300_200_N.mat
Saved fold 3 preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S10_E1_A1_fold3_300_200_N.mat
Saved fold 4 preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S10_E1_A1_fold4_300_200_N.mat
Saved fold 5 preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S10_E1_A1_fold5_300_200_N.mat
Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signa